# How should the wavelets be normalised before clustering?

The pipeline z-scores each session **within itself** before the k=8 paw clustering
(`session_zscored_supersession_wavelets_paw08-19-2026`, then a second global z-score in
`describe_states.py:22`). The stated reason was to make the two cameras comparable. This
notebook asks what that choice costs and what the alternatives buy.

### The three schemes

These are three ways of normalising the **wavelet matrix before clustering** — not anything
derived from syllable use.

| scheme | what it does | what it removes |
|---|---|---|
| `session` | z-score each of the 20 columns across that session's frames | each session's own level **and scale**, per feature |
| `rig` | z-score each column across all frames of a **lab** | the lab's level and scale — i.e. the rig |
| `shape` | divide each **frame** by the sum of its own 20 values | that frame's total magnitude, leaving only the profile across (paw × axis × frequency) |

`amplitude` is **not** a normalisation — it is an extra *feature*: `log` of the session's mean
raw wavelet power, one number per session, appended as a 9th column beside the 8 occupancies.

### How a per-session feature vector is built

The clustering assigns every **frame** to one of 8 states. A session's feature vector is its
**occupancy** — the fraction of its frames in each state. That is the same construction the
syllable pipeline calls syllable use, but computed here on the subsampled frames rather than on
the binned trial files.

### Three questions

1. Does a session-level **laterality trait** survive the normalisation at all?
2. Can a state mean **"a lot of movement"** in absolute terms, the same for every animal?
3. Are mice **identifiable** — and identifiable **within one rig**, where rig signature carries
   no information?

## The data — and what it is not

Fitting is on `data/subsampled_wavelets/*.npy`: **189 files, one per session, 2000 frames ×
32 columns each**, raw un-z-scored power, written 2026-04-27.

- 20 of the 32 columns are used — the 5-frequency set (`l/r_paw × x/y × 0.5–8 Hz`). The 0.25,
  16 and 32 Hz bands and the 4 raw position columns are dropped, as `describe_states.py` does.
- **378,000 rows total.** A full session is ~70 min × 60 Hz ≈ 252,000 frames, so 2000 frames is
  roughly a **1% subsample**. The published fit used the same construction: 664,000 rows =
  332 sessions × 2000.
- **These frames carry no trial index and no timestamps.** That matters below.

The supersession the published fit actually used is not in the repo, so it is rebuilt here. The
rebuild reproduces that fit closely — 14 PCA components as published, median per-state profile
r = 0.96 under one-to-one assignment — but it is a **subset (189 of 332)**, so absolute numbers
here are not the published ones.

## Method

**mouse ID** — Leave-one-session-out over the cohort scored. For each held-out session the rest
are subsampled to at most 2 sessions per mouse, so a 7-session animal cannot set the
within-class scatter; LDA `solver='eigen'`, `shrinkage=0.5` (8–9 features against ~2 sessions
per class is far too thin unregularised), uniform priors. 20 draws, averaged. Mice with <2
sessions dropped. Chance = 1/n_mice of that cohort.

**ICC** — pingouin `ICC(1,1)`, single-measure one-way random effects, on the first 2 sessions of
each mouse that has 2 (pingouin needs a balanced table). An earlier version of this analysis
used a hand-rolled formula that returned something near `ICC(1,k)` and overstated
single-session repeatability by ~25%.

**laterality** — the RAW difference `mean(l_paw) − mean(r_paw)` per session, never
`LI = (L−R)/(|L|+|R|)`. After z-scoring both means are 0 by construction, so the ratio becomes
0/0 and reports large meaningless numbers.

In [ ]:
import glob
import os
import pathlib
import sys

import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# .../paper-individuality/ -- found by walking up from the working directory, so this
# works from a notebook (no __file__) and from any subfolder
_p = pathlib.Path.cwd().resolve()
while not (_p / 'paper_style.py').exists() and _p != _p.parent:
    _p = _p.parent
ROOT = str(_p) + '/'
K, SEED = 8, 2024
FRQ = ['0.5', '1.0', '2.0', '4.0', '8.0']
VAR_INIT = (['l_paw_x', 'l_paw_y', 'r_paw_x', 'r_paw_y']
            + [f'{p}_paw_{ax}{f}' for p in 'lr' for ax in 'xy'
               for f in ['0.5', '1.0', '2.0', '4.0', '8.0', '16.0', '32.0']])
VAR = ([f'l_paw_x{f}' for f in FRQ] + [f'l_paw_y{f}' for f in FRQ]
       + [f'r_paw_x{f}' for f in FRQ] + [f'r_paw_y{f}' for f in FRQ])
LI_IDX = [VAR.index(v) for v in VAR if v.startswith('l_paw')]
RI_IDX = [VAR.index(v) for v in VAR if v.startswith('r_paw')]
SCHEMES = ('session', 'rig', 'shape')

## Load, normalise, cluster

In [ ]:
def load():
    rows, sess, mouse = [], [], []
    for f in sorted(glob.glob(ROOT + 'data/subsampled_wavelets/*.npy')):
        a = np.load(f)
        if a.shape[1] != len(VAR_INIT):
            continue
        stem = os.path.basename(f)[:-4]
        rows.append(pd.DataFrame(a, columns=VAR_INIT)[VAR].to_numpy())
        sess += [stem[:36]] * len(a)
        mouse += [stem[37:]] * len(a)
    X = np.vstack(rows)
    ok = ~np.isnan(X).any(axis=1)
    return X[ok], np.array(sess)[ok], np.array(mouse)[ok]


def zscore_by(X, groups):
    out = np.empty_like(X)
    for g in np.unique(groups):
        m = groups == g
        out[m] = stats.zscore(X[m], axis=0, nan_policy='omit')
    return out


def normalise(X, sess, labs, mode):
    if mode == 'session':
        return zscore_by(X, sess)
    if mode == 'rig':
        return zscore_by(X, labs)
    if mode == 'shape':
        tot = X.sum(axis=1, keepdims=True)
        tot[tot <= 0] = 1.0
        return X / tot
    raise ValueError(mode)


def cluster(Z):
    """describe_states.py's recipe: global z-score, PCA to 95% var, KMeans k=8."""
    Zg = stats.zscore(Z, axis=0)
    pca = PCA(n_components=min(20, Zg.shape[1]))
    P = pca.fit_transform(Zg)
    k = min(Zg.shape[1], np.where(np.cumsum(pca.explained_variance_ratio_) >= 0.95)[0][0] + 1)
    return KMeans(n_clusters=K, random_state=SEED, n_init=10).fit_predict(P[:, :k]), k


def occupancy(lab, sess):
    d = pd.DataFrame({'s': sess, 'k': lab})
    return (d.groupby(['s', 'k']).size().unstack(fill_value=0)
            .reindex(columns=range(K), fill_value=0).pipe(lambda t: t.div(t.sum(1), axis=0)))


def mouse_id(tab, mouse_of, n_repeats=20, seed=0, cap=2):
    y = pd.factorize(pd.Index([mouse_of[s] for s in tab.index]))[0]
    Xf = tab.to_numpy(float)
    keep = np.isin(y, [m for m in np.unique(y) if (y == m).sum() >= 2])
    Xf, y = Xf[keep], y[keep]
    if len(np.unique(y)) < 2:
        return float('nan'), len(np.unique(y))
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_repeats):
        hit = []
        for t in range(len(Xf)):
            tr = np.setdiff1d(np.arange(len(Xf)), t)
            sel = np.concatenate([rng.choice(np.where(y[tr] == m)[0],
                                             min(cap, (y[tr] == m).sum()), replace=False)
                                  for m in np.unique(y[tr])])
            if y[t] not in y[tr][sel]:
                continue
            k = len(np.unique(y[tr][sel]))
            f = LinearDiscriminantAnalysis(solver='eigen', shrinkage=0.5,
                                           priors=np.ones(k) / k).fit(Xf[tr][sel], y[tr][sel])
            hit.append(f.predict(Xf[t:t + 1])[0] == y[t])
        if hit:
            out.append(np.mean(hit))
    return (float(np.mean(out)) if out else float('nan')), len(np.unique(y))


def icc11(values, mouse_of):
    """ICC(1,1) on the first 2 sessions of every mouse that has 2."""
    d = pd.DataFrame({'v': values, 'mouse': [mouse_of[s] for s in values.index]})
    d = d[d.groupby('mouse')['v'].transform('size') >= 2]
    d['rep'] = d.groupby('mouse').cumcount()
    b = d[d.rep < 2]
    b = b[b.groupby('mouse')['v'].transform('size') == 2]
    t = pg.intraclass_corr(data=b, targets='mouse', raters='rep', ratings='v')
    r = t[t['Type'] == 'ICC(1,1)'].iloc[0]
    return float(r['ICC']), np.asarray(r['CI95'], float), int(b.mouse.nunique())


def build():
    X, sess, mouse = load()
    mouse_of = dict(zip(sess, mouse))
    uniq = pd.unique(sess)
    sys.path.insert(0, ROOT + '4_mice')
    sys.path.insert(0, ROOT + 'learning_individuality')
    from functions import lab_labels
    raw = lab_labels(pd.Index(uniq),
                     mouse_names=pd.Series([mouse_of[s] for s in uniq], index=uniq))
    lab_of = {s: (l if isinstance(l, str) else '?') for s, l in zip(uniq, raw)}
    labs = np.array([lab_of[s] for s in sess])

    out = dict(X=X, sess=sess, mouse=mouse, uniq=uniq, mouse_of=mouse_of,
               lab_of=lab_of, labs=labs)
    out['raw_power'] = pd.Series({s: X[sess == s].mean() for s in uniq})
    out['amp'] = np.log(out['raw_power'])
    out['gap_raw'] = pd.Series({s: X[sess == s][:, LI_IDX].mean()
                                - X[sess == s][:, RI_IDX].mean() for s in uniq})
    fits = {}
    for mode in SCHEMES:
        Z = normalise(X, sess, labs, mode)
        lab, npc = cluster(Z)
        gap = pd.Series({s: Z[sess == s][:, LI_IDX].mean() - Z[sess == s][:, RI_IDX].mean()
                         for s in uniq})
        # every state described in RAW units, so the three fits are comparable
        # CENTROID, not just vigor and gap. Two states can sit on top of each other in the
        # (gap, vigor) plane and still be different movements -- under `rig`, states 4 and 6
        # match on both (16.4/16.8 vigor, -9.1/-10.5 gap) but 4 peaks at 2-4 Hz while 6
        # climbs to 8 Hz. Without this the plane silently merges them.
        fnum = np.array([float(f) for f in FRQ])
        cen = []
        for k in range(K):
            pw = np.array([X[lab == k][:, [VAR.index(f'{p}_paw_{ax}{f}')
                                           for p in 'lr' for ax in 'xy']].mean()
                           for f in FRQ])
            pw = np.clip(pw - pw.min(), 1e-9, None)
            cen.append(float((pw * fnum).sum() / pw.sum()))
        prof = pd.DataFrame({'vigor': [X[lab == k].mean() for k in range(K)],
                             'gap': [X[lab == k][:, LI_IDX].mean()
                                     - X[lab == k][:, RI_IDX].mean() for k in range(K)],
                             'centroid': cen,
                             'occ': [float((lab == k).mean()) for k in range(K)]})
        fits[mode] = dict(labels=lab, npc=npc, occ=occupancy(lab, sess),
                          gap_after=gap, prof=prof)
    out['fits'] = fits
    return out

In [ ]:
D = build()
print(f"{D['X'].shape[0]:,} rows, {len(D['uniq'])} sessions, "
      f"{len(set(D['mouse']))} mice, {len(set(D['lab_of'].values()))} labs")

## The three questions

In [ ]:
def report(D):
    X, sess, uniq, mouse_of = D['X'], D['sess'], D['uniq'], D['mouse_of']
    print(f"{X.shape[0]} rows, {len(uniq)} sessions, {len(set(D['mouse']))} mice, "
          f"{len(set(D['lab_of'].values()))} labs")
    print(f"session raw power {D['raw_power'].min():.1f}-{D['raw_power'].max():.1f} "
          f"({D['raw_power'].max() / D['raw_power'].min():.1f}x spread)")
    i, ci, n = icc11(D['gap_raw'], mouse_of)
    print(f"raw L-R gap: mean {D['gap_raw'].mean():+.3f} (right-biased), "
          f"sd {D['gap_raw'].std():.3f}, ICC(1,1) {i:.3f} CI{np.round(ci, 2)} over {n} mice\n")

    print('=' * 76)
    print('1. DOES A SESSION-LEVEL LATERALITY TRAIT SURVIVE THE NORMALISATION?')
    print('=' * 76)
    print(f'{"scheme":<10}{"sd of L-R gap":>15}{"ICC(1,1)":>10}{"CI95":>16}   verdict')
    for m in SCHEMES:
        g = D['fits'][m]['gap_after']
        if g.std() < 1e-9:
            print(f'{m:<10}{g.std():15.4f}{"--":>10}{"--":>16}   DESTROYED exactly')
            continue
        i, ci, _ = icc11(g, mouse_of)
        print(f'{m:<10}{g.std():15.4f}{i:10.3f}{str(np.round(ci, 2)):>16}   '
              f'{"preserved" if i > 0.4 else "weak"}')
    print('  per-session z-scores every COLUMN to mean 0 within the session, so l_paw and')
    print('  r_paw both become 0 and their difference is identically 0. Not small -- zero.')

    print('\n' + '=' * 76)
    print('2. CAN A STATE MEAN "A LOT OF MOVEMENT" FOR EVERY ANIMAL?')
    print('=' * 76)
    print(f'{"scheme":<10}{"PCA":>5}{"top state":>11}{"its vigor":>11}'
          f'{"occ~raw r":>11}{"p":>10}')
    for m in SCHEMES:
        f = D['fits'][m]
        top = int(f['prof']['vigor'].idxmax())
        r, p = stats.pearsonr(f['occ'][top].reindex(D['raw_power'].index), D['raw_power'])
        print(f'{m:<10}{f["npc"]:5d}{top:11d}{f["prof"]["vigor"][top]:11.2f}{r:11.3f}{p:10.2g}')
    print('  occ~raw r: does a session that MOVED more spend more time in the most vigorous')
    print('  state? Near zero means the state cannot tell a lively mouse from a quiet one.')

    print('\n' + '=' * 76)
    print('3. ARE MICE IDENTIFIABLE -- AND IDENTIFIABLE WITHIN ONE RIG?')
    print('=' * 76)
    lab_s = pd.Series(D['lab_of'])
    feats = {}
    for m in SCHEMES:
        feats[f'{m} occupancy'] = D['fits'][m]['occ']
    feats['shape occ + amplitude'] = D['fits']['shape']['occ'].assign(
        amp=D['amp'].reindex(D['fits']['shape']['occ'].index))
    print(f'{"features":<24}{"cohort":<12}{"mice":>6}{"acc":>8}{"chance":>8}{"x chance":>10}')
    rows = {}
    for name, tab in feats.items():
        a, n = mouse_id(tab, mouse_of)
        accs, ns, wts = [], [], []
        for l in sorted(set(lab_s)):
            idx = [s for s in tab.index if lab_s.get(s, '?') == l]
            if len(idx) < 6:
                continue
            a2, n2 = mouse_id(tab.loc[idx], mouse_of)
            if np.isnan(a2) or n2 < 3:
                continue
            accs.append(a2); ns.append(n2); wts.append(len(idx))
        w = np.array(wts, float); w /= w.sum()
        pooled = float(np.sum(w * np.array(accs)))
        chance = float(np.sum(w * (1 / np.array(ns))))
        rows[name] = (a, 1 / n, pooled, chance)
        print(f'{name:<24}{"all labs":<12}{n:6d}{a:8.3f}{1 / n:8.3f}{a * n:10.1f}')
        print(f'{"":<24}{"within lab":<12}{int(np.mean(ns)):6d}{pooled:8.3f}{chance:8.3f}'
              f'{pooled / chance:10.1f}   ({len(accs)} labs)')
    return rows

In [ ]:
rows = report(D)

## Summary figure

(a) laterality survival, (b) absolute vigor, (c) mouse vs rig, (d) where the variance lives,
(e) the states each scheme produces, in raw units so the three are comparable.

In [ ]:
def figures(D, rows, out=None):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    sys.path.insert(0, ROOT)
    import paper_style as ps
    ps.use('paper')

    TONE = {'session': '#9AA0A6', 'rig': ps.PAW_LEFT, 'shape': ps.PAW_RIGHT}
    fig = plt.figure(figsize=(ps.FIG['double'][0], 6.4))
    gs = fig.add_gridspec(3, 3, hspace=0.62, wspace=0.34)

    # --- (a) laterality survival ------------------------------------------------------
    ax = fig.add_subplot(gs[0, 0])
    vals, err = [], []
    for m in SCHEMES:
        g = D['fits'][m]['gap_after']
        if g.std() < 1e-9:
            vals.append(0.0); err.append((0, 0)); continue
        i, ci, _ = icc11(g, D['mouse_of'])
        vals.append(i); err.append((i - ci[0], ci[1] - i))
    ax.bar(range(3), vals, color=[TONE[m] for m in SCHEMES],
           yerr=np.array(err).T, capsize=2, edgecolor='#333', linewidth=.5)
    ax.set_xticks(range(3)); ax.set_xticklabels(SCHEMES)
    ax.set_ylabel('ICC(1,1) of the L-R gap')
    ax.set_title('(a) does a laterality trait survive?', loc='left')
    ax.text(0, .04, 'exactly 0', ha='center', fontsize=plt.rcParams['font.size'] * .8)
    ax.axhline(0, color='#333', lw=.6)

    # --- (b) absolute vigor -----------------------------------------------------------
    ax = fig.add_subplot(gs[0, 1:])
    for m in SCHEMES:
        f = D['fits'][m]
        top = int(f['prof']['vigor'].idxmax())
        x = D['raw_power']; y = f['occ'][top].reindex(x.index)
        r = stats.pearsonr(x, y)[0]
        ax.scatter(x, y, s=5, alpha=.55, color=TONE[m], lw=0,
                   label=f'{m}  r={r:+.2f}')
        b = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 20)
        ax.plot(xs, np.polyval(b, xs), color=TONE[m], lw=1.4)
    ax.set_xlabel("session's RAW mean wavelet power")
    ax.set_ylabel('occupancy of the\nmost vigorous state')
    ax.set_title('(b) can a state mean "a lot of movement" for everyone?', loc='left')
    ax.legend(fontsize=plt.rcParams['font.size'] * .8, loc='upper right')

    # --- (c) mouse ID, all labs vs within lab -----------------------------------------
    ax = fig.add_subplot(gs[1, :2])
    names = list(rows); x = np.arange(len(names)); w = .36
    ax.bar(x - w / 2, [rows[n][0] / rows[n][1] for n in names], w, label='all labs',
           color='#C6CBD1', edgecolor='#333', linewidth=.5)
    ax.bar(x + w / 2, [rows[n][2] / rows[n][3] for n in names], w, label='within one lab',
           color=ps.PAW_LEFT, edgecolor='#333', linewidth=.5)
    ax.axhline(1, color='#333', lw=.8, ls='--')
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' occupancy', '').replace('shape occ + amplitude',
                                                            'shape + amp') for n in names],
                       rotation=18, ha='right')
    ax.set_ylabel('accuracy / chance')
    ax.set_title('(c) is it the mouse or the rig?', loc='left')
    ax.legend(fontsize=plt.rcParams['font.size'] * .8)

    # --- (d) where the variance lives -------------------------------------------------
    ax = fig.add_subplot(gs[1, 2])
    labs = pd.Series(D['lab_of'])
    mice = pd.Series({s: D['mouse_of'][s] for s in D['uniq']})
    share = []
    for v in (D['gap_raw'], D['amp']):
        tot = ((v - v.mean()) ** 2).sum()
        share.append([sum(len(g) * (g.mean() - v.mean()) ** 2 for _, g in v.groupby(k)) / tot
                      for k in (labs, mice)])
    share = np.array(share)
    ax.bar([0, 1], share[:, 0] * 100, .5, label='between labs',
           color='#C6CBD1', edgecolor='#333', linewidth=.5)
    ax.bar([0, 1], (share[:, 1] - share[:, 0]) * 100, .5, bottom=share[:, 0] * 100,
           label='between mice', color=ps.PAW_LEFT, edgecolor='#333', linewidth=.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['L-R gap', 'amplitude'])
    ax.set_ylabel('% of variance'); ax.set_ylim(0, 100)
    ax.set_title('(d) lab vs mouse', loc='left')
    ax.legend(fontsize=plt.rcParams['font.size'] * .75, loc='lower right')

    # --- (e) the states themselves, in RAW units --------------------------------------
    for j, m in enumerate(SCHEMES):
        ax = fig.add_subplot(gs[2, j])
        p = D['fits'][m]['prof']
        # y is the SPECTRAL CENTROID, not vigor: under `rig` states 4 and 6 have the same
        # vigor and the same gap and were drawn on top of each other, their labels
        # overprinting into one blob. They differ in frequency, so this axis separates them.
        ax.scatter(p['gap'], p['centroid'], s=40 + 900 * p['occ'], color=TONE[m],
                   edgecolor='#333', linewidth=.5, alpha=.85)
        for k in range(K):
            ax.annotate(str(k), (p['gap'][k], p['centroid'][k]), fontsize=6,
                        ha='center', va='center', color='white', weight='bold')
        ax.axvline(0, color='#999', lw=.6, ls='--')
        ax.set_xlabel('state L-R gap (raw)')
        if j == 0:
            ax.set_ylabel('state spectral centroid (Hz)')
        ax.set_title(f'(e{j + 1}) {m} states', loc='left')
    fig.suptitle('Normalisation choice: what each scheme keeps and loses', y=0.985)
    out = out or (ROOT + 'figures/paw_normalisation_choice.png')
    fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'\nwrote {out}')
    return fig

In [ ]:
figures(D, rows)

## The states each scheme produces

The panels `syllable_composition.ipynb` draws, repeated per cluster space: state
characterisation (vigor / gap / occupancy) and the per-paw spectrum.

**The trial-aligned composition histogram is missing on purpose.** The subsampled wavelet files
carry no trial index and no timestamps, so there is nothing to align to. Producing it would mean
propagating each new clustering back to the full recordings and rebuilding the binned syllable
files — a much larger job than this comparison.

Each fit is coloured from **its own** (gap, vigor) by `paper_style`'s rule, because KMeans labels
are arbitrary: state 3 under one scheme is not state 3 under another.

In [ ]:
def state_palette(prof, h_left=310, h_right=90, c_max=0.205, l_hi=0.93, l_lo=0.30):
    """paper_style's paw rule applied to a NEW clustering's own states.

    Hue from the sign of the state's left-right gap, chroma linear in |gap|, lightness from
    vigor. The published PAW_STATE_COLORS cannot be reused directly because KMeans labels are
    arbitrary -- state 3 under one scheme is not state 3 under another -- so each fit is
    coloured from its own numbers and the three become readable in one visual language.
    """
    g = prof['gap'].to_numpy()
    v = prof['vigor'].to_numpy()
    cn = np.abs(g) / (np.abs(g).max() or 1.0)
    lin = (v - v.min()) / ((v.max() - v.min()) or 1.0)
    rank = stats.rankdata(v, method='ordinal') - 1
    t = 0.66 * lin ** 0.6 + 0.34 * (rank / max(rank.max(), 1))
    out = []
    for i in range(len(g)):
        L, C = l_hi - t[i] * (l_hi - l_lo), 0.018 + c_max * cn[i]
        h = np.radians(h_left if g[i] >= 0 else h_right)
        a, b = C * np.cos(h), C * np.sin(h)
        l_, m_, s_ = ((L + 0.3963377774 * a + 0.2158037573 * b) ** 3,
                      (L - 0.1055613458 * a - 0.0638541728 * b) ** 3,
                      (L - 0.0894841775 * a - 1.2914855480 * b) ** 3)
        rgb = np.clip([4.0767416621 * l_ - 3.3077115913 * m_ + 0.2309699292 * s_,
                       -1.2684380046 * l_ + 2.6097574011 * m_ - 0.3413193965 * s_,
                       -0.0041960863 * l_ - 0.7034186147 * m_ + 1.7076147010 * s_], 0, 1)
        e = np.where(rgb <= 0.0031308, 12.92 * rgb, 1.055 * rgb ** (1 / 2.4) - 0.055)
        out.append('#%02X%02X%02X' % tuple(np.round(255 * np.clip(e, 0, 1)).astype(int)))
    return out

In [ ]:
def scheme_figure(D, mode, out=None):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    sys.path.insert(0, ROOT)
    import paper_style as ps
    ps.use('paper')

    X, lab = D['X'], D['fits'][mode]['labels']
    prof = D['fits'][mode]['prof']
    order = list(np.argsort(prof['vigor'].to_numpy()))      # pale-still -> dark-vigorous
    cols = state_palette(prof)
    ypos = -np.arange(K)

    fig = plt.figure(figsize=(ps.FIG['double'][0], 5.4))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.15], hspace=0.55, wspace=0.30)
    for j, (key, title, div) in enumerate([
            ('vigor', 'Vigor\nmean RAW wavelet power', False),
            ('gap', 'Left - right gap\nRAW units', True),
            ('centroid', 'Spectral centroid\nHz -- slow vs fast', False)]):
        ax = fig.add_subplot(gs[0, j])
        vals = [prof[key][s] for s in order]
        ax.barh(ypos, vals, height=.74, color=[cols[s] for s in order],
                edgecolor='#333', linewidth=.5)
        if div:
            ax.axvline(0, color='#333', lw=.8)
        ax.set_title(title, fontsize=plt.rcParams['font.size'] * .95)
        ax.spines[['top', 'right']].set_visible(False)
        if j == 0:
            ax.set_yticks(ypos); ax.set_yticklabels([f'state {s}' for s in order],
                                                    fontsize=plt.rcParams['font.size'] * .85)
        else:
            ax.set_yticks([])
        if div:
            ax.set_xlabel('← right          left →',
                          fontsize=plt.rcParams['font.size'] * .8, color='#666')

    # the per-paw spectrum, exactly the notebook's second figure
    x = np.arange(len(FRQ))
    lo = hi = None
    for j, (p, name) in enumerate([('l', 'left paw'), ('r', 'right paw')]):
        ax = fig.add_subplot(gs[1, j])
        for s in order:
            y = [X[lab == s][:, [VAR.index(f'{p}_paw_x{f}'), VAR.index(f'{p}_paw_y{f}')]].mean()
                 for f in FRQ]
            ax.plot(x, y, color=cols[s], lw=2.2, marker='o', ms=3.5,
                    markeredgecolor='#33333366', markeredgewidth=.4)
            ax.annotate(str(s), (x[-1], y[-1]), xytext=(4, 0), textcoords='offset points',
                        va='center', fontsize=plt.rcParams['font.size'] * .7, color=cols[s])
            lo = y[0] if lo is None else min(lo, min(y)); hi = max(hi or 0, max(y))
        ax.set_xticks(x); ax.set_xticklabels(FRQ)
        ax.set_xlim(-.25, len(FRQ) - .55)
        ax.set_xlabel('wavelet frequency (Hz)')
        ax.set_title(name, fontsize=plt.rcParams['font.size'],
                     color=ps.PAW_LEFT if p == 'l' else ps.PAW_RIGHT, loc='left')
        ax.spines[['top', 'right']].set_visible(False)
        if j == 0:
            ax.set_ylabel('mean RAW power')
    for a in fig.axes[-2:]:
        a.set_ylim(lo - .05 * hi, hi * 1.08)

    # VIGOR against CENTROID: the gap x vigor plane merged states that differ only in
    # frequency (rig 4 and 6), so the second axis is the spectrum, and hue still carries the
    # gap. Area is occupancy.
    ax = fig.add_subplot(gs[1, 2])
    ax.scatter(prof['centroid'], prof['vigor'], s=40 + 900 * prof['occ'], c=cols,
               edgecolor='#333', linewidth=.5)
    for s in range(K):
        ax.annotate(str(s), (prof['centroid'][s], prof['vigor'][s]), fontsize=6, ha='center',
                    va='center', color='white', weight='bold')
    ax.set_xlabel('spectral centroid (Hz)'); ax.set_ylabel('state vigor (raw)')
    ax.set_title('slow/fast vs how much, colour = side',
                 fontsize=plt.rcParams['font.size'] * .9, loc='left')
    ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle(f"'{mode}' normalisation — the 8 states it produces", y=0.98)
    out = out or (ROOT + f'figures/paw_states_{mode}.png')
    fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'  wrote {os.path.basename(out)}')

In [ ]:
for m in SCHEMES:
    scheme_figure(D, m)

## What do the discriminant dimensions capture?

The cheap version of the LDA test: fit LDA with **mouse as the class** on the per-session
feature vector (the 8 occupancies, plus log-amplitude for the `shape + amp` variant), then ask
what each discriminant is made of.

This is not the paper's LDA. That one runs on the 360-feature binned syllable matrix, which
needs trial-aligned state labels — and the subsampled wavelet files carry no trial index. Doing
it properly would mean propagating each clustering back to the full recordings and rebuilding
the binned syllable files, on the **59 sessions** that have both full wavelets and an embedding
rather than the pipeline's 260.

Each discriminant is read two ways:

- **loadings** — which states it weights, and what those states are (from their gap and centroid)
- **correlates** — how the session scores line up with three interpretable quantities: the
  session's raw power (amplitude), its left−right gap (laterality), and its mean spectral
  centroid (slow vs fast). This is what says "LD1 is amplitude" rather than "LD1 is state 3
  minus state 5".

Finally each LD is correlated with the **published** LD1/LD2/LD3 on the sessions the two share,
to see whether any of this resembles the axis the paper already reports.

In [ ]:
"""CHEAP LDA -- mouse as class, on the per-session occupancy vectors."""
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# session-level quantities to interpret the discriminants against
FNUM = np.array([float(f) for f in FRQ])
sess_centroid = pd.Series({
    s: float((lambda pw: (np.clip(pw - pw.min(), 1e-9, None) * FNUM).sum()
              / np.clip(pw - pw.min(), 1e-9, None).sum())(
        np.array([D['X'][D['sess'] == s][:, [VAR.index(f'{p}_paw_{ax}{f}')
                                             for p in 'lr' for ax in 'xy']].mean()
                  for f in FRQ])))
    for s in D['uniq']})
CORRELATES = {'amplitude  (raw power)': D['raw_power'],
              'laterality (L-R gap)': D['gap_raw'],
              'speed      (centroid Hz)': sess_centroid}

emb_path = ROOT + 'clustering/data_files/mouse_LDA_5_bins_raw_shrink0.5_360_25-09-2026'
emb = pd.read_pickle(emb_path).set_index('session') if os.path.exists(emb_path) else None

FEATS = {m: D['fits'][m]['occ'] for m in SCHEMES}
FEATS['shape + amp'] = D['fits']['shape']['occ'].assign(
    amp=D['amp'].reindex(D['fits']['shape']['occ'].index))

for name, tab in FEATS.items():
    y = pd.factorize(pd.Index([D['mouse_of'][s] for s in tab.index]))[0]
    keep = np.isin(y, [m for m in np.unique(y) if (y == m).sum() >= 2])
    Xf, yy, idx = tab.to_numpy(float)[keep], y[keep], tab.index[keep]
    k = len(np.unique(yy))
    # fit on ALL sessions of the kept mice, shrinkage as elsewhere; this is for reading the
    # structure, not for scoring -- the scored version is the leave-one-out above
    lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=0.5,
                                     priors=np.ones(k) / k).fit(Xf, yy)
    Z = (Xf - lda.means_.mean(0)) @ lda.scalings_[:, :4]

    print('=' * 74)
    print(f'{name}   ({Xf.shape[1]} features, {k} mice, {len(Xf)} sessions)')
    print('=' * 74)
    print(f'{"":>5}' + ''.join(f'{c:>11}' for c in list(CORRELATES) + ['pub LD1', 'pub LD3']))
    for d in range(min(4, Z.shape[1])):
        cells = []
        for v in CORRELATES.values():
            cells.append(f'{stats.pearsonr(Z[:, d], v.reindex(idx))[0]:+11.2f}')
        for pub_d in (0, 2):
            if emb is None:
                cells.append(f'{"--":>11}'); continue
            sh = [s for s in idx if s in emb.index]
            if len(sh) < 10:
                cells.append(f'{"--":>11}'); continue
            m_ = np.isin(idx, sh)
            cells.append(f'{stats.pearsonr(Z[m_, d], emb.loc[idx[m_], pub_d])[0]:+11.2f}')
        print(f'  LD{d + 1}' + ''.join(cells))
    # the states each discriminant leans on, with what those states ARE
    if name != 'shape + amp':
        prof = D['fits'][name if name in SCHEMES else 'shape']['prof']
        for d in range(2):
            w = lda.scalings_[:, d]
            top = np.argsort(-np.abs(w))[:3]
            bits = [f'state {t} (w{w[t]:+.1f}, gap {prof["gap"][t]:+.0f}, '
                    f'{prof["centroid"][t]:.1f} Hz)' for t in top]
            print(f'  LD{d + 1} leans on: ' + ';  '.join(bits))
    print()
